# Lab Activity 5: Classification — Linear Models & Neural Networks
**Course:** CSE473: Computational Intelligence — Mechatronics Engineering and Automation Program
**Prepares you for:** Lab Assignment 05 — Double Moon Dataset, Binary Classification

## 🎯 Learning objectives
By the end of this lab you will be able to:
- Generate the **double moon** dataset geometrically (half-annulus crescents, reproducibly)
- Label, shuffle and split data into train / validation / test with numpy
- Train a **logistic regression** classifier by full-batch gradient descent and read its loss curve
- Train a **pure-numpy neural network** (one hidden layer, sigmoid, MSE, momentum)
- Compare both models by accuracy and decision boundary — the assignment's final task

⏱ **Estimated time: ~55 minutes**

## How this lab works
- The lab is split into **Parts**; each Part teaches one topic.
- Each Part starts with a short explanation plus a runnable **✏️ Worked example** — run it, tweak it, break it.
- Then you solve **🎯 Problems**. Read the task, write your code in the starter cell, and try it before opening any hints.
- Stuck? Open the **💡 Hint** blocks below each problem — Hint 1 is a nudge, Hint 2 names the approach. They get more specific as you go.
- Verify yourself with the **🧪 Self-check** cells — they run deterministic checks and fail with guiding messages until your solution is right.
- Truly stuck? The **✅ Reveal solution** block at the bottom of each hint section shows full working code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Setup OK — NumPy and matplotlib are ready.")

## Part 1: One moon, geometrically (≈12 min)

The **double moon** dataset is two interleaved crescents. Each moon is a **half-annulus**: radii between `radius - width/2` and `radius + width/2`, angles $\theta \in (0, \pi)$.

- Opening **up** from center $(c_x, c_y)$:  $x = c_x + r\cos\theta$, $y = c_y + r\sin\theta$  (all $y \geq c_y$)
- Opening **down**: negate both —  $x = c_x - r\cos\theta$, $y = c_y - r\sin\theta$  (all $y \leq c_y$)

The assignment's moons: red (label 0) opens up at $(0, 0)$; blue (label 1) opens down at $(\text{radius}, d)$ — so the crescents interleave. Draw radii and angles from a **seeded** generator so your data (and your grades) are reproducible.

In [ ]:
# --- Worked example: build one crescent and inspect its shape ---
rng = np.random.default_rng(0)
r = 10 - 3 + 6 * rng.random(300)     # radii in [radius - width/2, radius + width/2] = [7, 13]
theta = np.pi * rng.random(300)      # angles in (0, pi)

xs = r * np.cos(theta)
ys = r * np.sin(theta)               # opening up at (0, 0): y >= 0

print("radial range:", round(float(np.hypot(xs, ys).min()), 2), "to", round(float(np.hypot(xs, ys).max()), 2))
print("all y >= 0?", bool((ys >= 0).all()))

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(xs, ys, s=8, c="tomato")
ax.set_title("one moon: upper half-annulus at (0, 0)")
ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.show()

## 🎯 Problem 5.1 — Generate one moon

**Given:** a count `n`, a `center`, an outer `radius`, a radial `width`, an opening direction and a `seed`.

**Required:** write `generate_moon(n, center=(0.0, 0.0), radius=10.0, width=6.0, downward=False, seed=0)` returning an `(n, 2)` array of points: a half-annulus as in the concept text, radii uniform in $[\text{radius} - \text{width}/2, \text{radius} + \text{width}/2]$, angles uniform in $(0, \pi)$, negated when `downward`.

**Expected output:** every point's distance from `center` lies in the radial band; upward moons have $y \geq c_y$, downward moons $y \leq c_y$; the same `seed` reproduces the same moon.

In [ ]:
def generate_moon(n, center=(0.0, 0.0), radius=10.0, width=6.0, downward=False, seed=0):
    """Generate one half-annulus (crescent) of points.

    Args:
        n: number of points
        center: (cx, cy) crescent center
        radius: outer radius (the band is centered on it)
        width: radial thickness of the annulus band
        downward: open downward (y <= cy) instead of upward (y >= cy)
        seed: reproducibility seed

    Returns:
        X: array of shape (n, 2)
    """
    # TODO: Your code here
    pass

# Demo call
demo_moon = generate_moon(300, seed=0)
if demo_moon is not None:
    print("moon shape:", demo_moon.shape)
    print("y range:", round(float(np.asarray(demo_moon)[:, 1].min()), 2), "to",
          round(float(np.asarray(demo_moon)[:, 1].max()), 2))
else:
    print("Implement generate_moon to power this demo.")

<details>
<summary>💡 Hint 1 — draw radii and angles</summary>

One seeded generator, two uniform draws: radii in [radius - width/2, radius + width/2] and angles in (0, pi). The point is (cx + r*cos, cy + r*sin) — flipping the sign of both coordinates turns an upward crescent downward.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
rng = np.random.default_rng(seed)
r = radius - width / 2 + width * rng.random(n)
theta = rng.random(n) * np.pi
sgn = -1.0 if downward else 1.0
xs = center[0] + sgn * r * np.cos(theta)
ys = center[1] + sgn * r * np.sin(theta)
return np.column_stack([xs, ys])
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def generate_moon(n, center=(0.0, 0.0), radius=10.0, width=6.0, downward=False, seed=0):
    """Generate one half-annulus (crescent) of points."""
    rng = np.random.default_rng(seed)
    r = radius - width / 2 + width * rng.random(n)
    theta = rng.random(n) * np.pi
    sgn = -1.0 if downward else 1.0
    xs = center[0] + sgn * r * np.cos(theta)
    ys = center[1] + sgn * r * np.sin(theta)
    return np.column_stack([xs, ys])
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.1
m1 = generate_moon(200, seed=1)
assert m1 is not None, "❌ generate_moon returned None — replace the 'pass'. See Hint 2 in Part 1."
m1 = np.asarray(m1, dtype=float)
assert m1.shape == (200, 2), f"❌ Expected shape (200, 2), got {m1.shape}."
d1 = np.hypot(m1[:, 0], m1[:, 1])
assert d1.min() >= 7.0 - 1e-9 and d1.max() <= 13.0 + 1e-9, "❌ All radii must lie in [radius - width/2, radius + width/2] = [7, 13]. Check the radius formula."
assert m1[:, 1].min() >= -1e-9, "❌ An upward moon (downward=False) must have y >= 0 — check the sin term."
m2 = np.asarray(generate_moon(200, downward=True, seed=1), dtype=float)
assert m2[:, 1].max() <= 1e-9, "❌ A downward moon must have y <= 0 — negate the coordinates (or shift the angle by pi)."
m3 = np.asarray(generate_moon(150, center=(5.0, 2.0), radius=8.0, width=2.0, seed=4), dtype=float)
d3 = np.hypot(m3[:, 0] - 5.0, m3[:, 1] - 2.0)
assert d3.min() >= 7.0 - 1e-9 and d3.max() <= 9.0 + 1e-9, "❌ The radial band must be measured from `center` with the given radius/width."
m1b = np.asarray(generate_moon(200, seed=1), dtype=float)
assert np.allclose(m1, m1b), "❌ Same seed must reproduce the same moon — pass seed into np.random.default_rng."
m1c = np.asarray(generate_moon(200, seed=2), dtype=float)
assert not np.allclose(m1, m1c), "❌ Different seeds should give different moons."
print("✅ Problem 5.1 passed — one crescent at a time.")

## 🎯 Problem 5.2 — The full double moon

**Given:** class sizes `N1`, `N2` and the usual moon parameters.

**Required:** write `generate_double_moon(N1, N2, radius=10, width=6, d=7, seed=42)` returning `(X, y)`:

- label 0: `N1` points, upward moon at $(0, 0)$,
- label 1: `N2` points, downward moon at $(\text{radius}, d)$,
- `y`: integer labels in `{0, 1}` of shape `(N1 + N2,)`.

**Expected output:** `X` stacks both moons (shape `(N1+N2, 2)`); every label-0 point is within the radial band of $(0, 0)$ with $y \geq 0$; every label-1 point is within the radial band of $(\text{radius}, d)$ with $y \leq d$. Use `seed` for moon 0 and `seed + 1` for moon 1 so both are reproducible.

In [ ]:
def generate_double_moon(N1, N2, radius=10, width=6, d=7, seed=42):
    """Generate the two-category double-moon dataset (interleaved crescents).

    Args:
        N1: points in class 0 (upward moon at (0, 0))
        N2: points in class 1 (downward moon at (radius, d))
        radius: outer radius of both moons
        width: radial thickness of both moons
        d: vertical offset of the second moon's center
        seed: reproducibility seed

    Returns:
        (X, y): X of shape (N1 + N2, 2); y in {0, 1} of shape (N1 + N2,)
    """
    # TODO: Your code here
    pass

# Demo call
demo_dm = generate_double_moon(200, 200)
if demo_dm is not None:
    Xd5, yd5 = demo_dm
    print("X shape:", Xd5.shape, "| class counts:", np.bincount(yd5))
else:
    print("Implement generate_double_moon to power this demo.")

<details>
<summary>💡 Hint 1 — two moons, two seeds</summary>

Call generate_moon twice — upward at (0, 0) with `seed`, downward at (radius, d) with `seed + 1` — then vstack the points and concatenate the label arrays (zeros then ones).
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
X0 = generate_moon(N1, center=(0.0, 0.0), downward=False, seed=seed)
X1 = generate_moon(N2, center=(radius, d), downward=True, seed=seed + 1)
X = np.vstack([X0, X1])
y = np.concatenate([np.zeros(N1), np.ones(N2)]).astype(int)
return X, y
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def generate_double_moon(N1, N2, radius=10, width=6, d=7, seed=42):
    """Generate the two-category double-moon dataset (interleaved crescents)."""
    X0 = generate_moon(N1, center=(0.0, 0.0), radius=radius, width=width,
                       downward=False, seed=seed)
    X1 = generate_moon(N2, center=(float(radius), float(d)), radius=radius, width=width,
                       downward=True, seed=seed + 1)
    X = np.vstack([X0, X1])
    y = np.concatenate([np.zeros(N1), np.ones(N2)]).astype(int)
    return X, y
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.2
data12 = generate_double_moon(150, 250, seed=42)
assert data12 is not None, "❌ generate_double_moon returned None — replace the 'pass'. See Hint 2 in Part 1."
X12 = np.asarray(data12[0], dtype=float); y12 = np.asarray(data12[1])
assert X12.shape == (400, 2) and y12.shape == (400,), f"❌ Expected X (400, 2) and y (400,), got {X12.shape} / {y12.shape}."
assert set(np.unique(y12)) == {0, 1}, "❌ Labels must be exactly {0, 1}."
assert (y12 == 0).sum() == 150 and (y12 == 1).sum() == 250, "❌ Class counts must match N1 / N2."
X0c, X1c = X12[y12 == 0], X12[y12 == 1]
r0 = np.hypot(X0c[:, 0], X0c[:, 1])
assert r0.min() >= 7.0 - 1e-9 and r0.max() <= 13.0 + 1e-9, "❌ Class 0 must live in the radial band of the (0, 0) moon."
assert X0c[:, 1].min() >= -1e-9, "❌ Class 0 is the UPWARD moon — its y must be >= 0."
r1 = np.hypot(X1c[:, 0] - 10.0, X1c[:, 1] - 7.0)
assert r1.min() >= 7.0 - 1e-9 and r1.max() <= 13.0 + 1e-9, "❌ Class 1 must live in the radial band of the (radius, d) = (10, 7) moon."
assert X1c[:, 1].max() <= 7.0 + 1e-9, "❌ Class 1 is the DOWNWARD moon — its y must be <= d."
data12b = generate_double_moon(150, 250, seed=42)
assert np.allclose(X12, np.asarray(data12b[0], dtype=float)) and np.array_equal(y12, np.asarray(data12b[1])), "❌ Same seed must reproduce the dataset."
print("✅ Problem 5.2 passed — the double moon is ready.")

## Part 2: Shuffle & split into train / validation / test (≈10 min)

Machine learning needs **held-out data**: train on one slice, tune on a second, grade on a third.

- `rng.permutation(len(X))` shuffles the row indices — always shuffle **before** splitting, and shuffle the *indices*, not the data, so rows keep their labels.
- Slice the shuffled indices into contiguous chunks: `[:n_train]`, `[n_train:n_train+n_val]`, and the rest is the test set.
- Default split: 60 / 20 / 20 (`train_frac=0.6`, `val_frac=0.2`).
- Everything stays reproducible through one seeded generator.

In [ ]:
# --- Worked example: split a tiny labeled dataset ---
X_toy = np.arange(20).reshape(10, 2).astype(float)
y_toy = np.array([0, 1] * 5)

rng = np.random.default_rng(3)
idx = rng.permutation(len(X_toy))
n_train, n_val = 6, 2
tr, va, te = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]
print("train idx:", tr, "| val idx:", va, "| test idx:", te)
print("every row lands in exactly one split:",
      sorted(np.concatenate([tr, va, te]).tolist()) == list(range(10)))
print("labels travelled with their rows:", y_toy[tr][:5], " <- y values AT the train indices")

## 🎯 Problem 5.3 — train_validation_test_split

**Given:** arrays `X`, `y` and fractions `train_frac`, `val_frac`.

**Required:** write `train_validation_test_split(X, y, train_frac=0.6, val_frac=0.2, seed=42)` shuffling the indices with `np.random.default_rng(seed)` and returning `((X_train, y_train), (X_val, y_val), (X_test, y_test))`, with `n_train = int(train_frac * len(X))` and `n_val = int(val_frac * len(X))`.

**Expected output:** the three splits are disjoint and together cover every row exactly once; with a balanced dataset each split keeps roughly both classes. This is the exact function **Lab Assignment 05** ships in its Task 1.

In [ ]:
def train_validation_test_split(X, y, train_frac=0.6, val_frac=0.2, seed=42):
    """Shuffle and split (X, y) into train / validation / test subsets.

    Args:
        X: feature array of shape (n, d)
        y: label array of shape (n,)
        train_frac: fraction of rows for training
        val_frac: fraction of rows for validation
        seed: reproducibility seed

    Returns:
        ((X_train, y_train), (X_val, y_val), (X_test, y_test))
    """
    # TODO: Your code here
    pass

# Demo call (uses your Problem 5.2 dataset once implemented)
demo_data = generate_double_moon(100, 100, seed=9)
if demo_data is not None:
    demo_X, demo_y = demo_data
    demo_splits = train_validation_test_split(demo_X, demo_y, seed=9)
    if demo_splits is not None:
        print("split sizes:", [len(s[0]) for s in demo_splits])
    else:
        print("Implement train_validation_test_split to see the sizes.")
else:
    print("Finish Problem 5.2 first to power this demo.")

<details>
<summary>💡 Hint 1 — shuffle indices, then slice</summary>

Never reorder X and y separately — shuffle ONE permutation of indices and slice it into three contiguous chunks. Rows and labels then move together by construction.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
rng = np.random.default_rng(seed)
idx = rng.permutation(len(X))
n_train = int(train_frac * len(X)); n_val = int(val_frac * len(X))
tr, va, te = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]
return (X[tr], y[tr]), (X[va], y[va]), (X[te], y[te])
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def train_validation_test_split(X, y, train_frac=0.6, val_frac=0.2, seed=42):
    """Shuffle and split (X, y) into train / validation / test subsets."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_train = int(train_frac * len(X))
    n_val = int(val_frac * len(X))
    tr, va, te = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]
    return (X[tr], y[tr]), (X[va], y[va]), (X[te], y[te])
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.3
Xt = np.arange(500).reshape(250, 2).astype(float)   # rows are unique -> leakage is detectable
yt = np.array([0, 1] * 125)
sp = train_validation_test_split(Xt, yt, seed=0)
assert sp is not None, "❌ train_validation_test_split returned None — replace the 'pass'. See Hint 2 in Part 2."
(Xtr5, ytr5), (Xv5, yv5), (Xte5, yte5) = sp
assert len(Xtr5) == 150 and len(Xv5) == 50 and len(Xte5) == 50, f"❌ Expected sizes 150/50/50, got {len(Xtr5)}/{len(Xv5)}/{len(Xte5)}."
seen = np.sort(np.vstack([Xtr5, Xv5, Xte5])[:, 0])
assert np.array_equal(seen, np.arange(0, 500, 2)), "❌ The three splits must be disjoint and cover every row exactly once — check your index slices."
sp2 = train_validation_test_split(Xt, yt, seed=0)
assert np.allclose(sp2[0][0], Xtr5), "❌ Same seed must reproduce the same split."
sp3 = train_validation_test_split(Xt, yt, seed=1)
assert not np.allclose(sp3[0][0], Xtr5), "❌ Different seeds should give different shuffles."
data23 = generate_double_moon(150, 150, seed=5)
assert data23 is not None, "❌ generate_double_moon returned None — finish Problem 5.2 first."
X23, y23 = data23
sp23 = train_validation_test_split(X23, y23, seed=5)
for i, (Xs_i, ys_i) in enumerate(sp23):
    assert len(np.unique(ys_i)) == 2, f"❌ Split {i} lost a class — shuffle the indices before slicing so each split stays roughly stratified."
    frac0 = float(np.mean(ys_i == 0))
    assert 0.3 <= frac0 <= 0.7, f"❌ Split {i} is heavily imbalanced (class-0 fraction {frac0:.2f}) — did you shuffle?"
print("✅ Problem 5.3 passed — clean, seeded, roughly stratified splits.")

## Part 3: The linear classifier (≈12 min)

A **logistic regression** scores each point $z = w^\top x + b$, squeezes it through the sigmoid $\sigma(z) = 1/(1 + e^{-z})$ into a probability, and fits $w, b$ by **full-batch gradient descent** on the cross-entropy loss:

$$L = -\tfrac{1}{n}\sum_i \left[y_i \log p_i + (1 - y_i)\log(1 - p_i)\right], \qquad \nabla_w L = \tfrac{1}{n}X^\top(p - y), \quad \nabla_b L = \tfrac{1}{n}\sum_i (p_i - y_i)$$

- Initialize $w = 0$, $b = 0$; repeat for `epochs`: forward → loss → gradients → step.
- **Record** the training loss every epoch (and the validation loss when validation data is given) — the loss curve diagnoses training, and the assignment asks for it.
- Predict with $\sigma(z) \geq 0.5$; the decision boundary $z = 0$ is a **straight line** — fine for tilted blobs, limited for crescents.

In [ ]:
# --- Worked example: logistic regression on separable toy data, step by step ---
rng = np.random.default_rng(1)
Xa = rng.normal((-1.5, -1.5), 0.6, size=(80, 2))
Xb = rng.normal((1.5, 1.5), 0.6, size=(80, 2))
Xtoy = np.vstack([Xa, Xb])
ytoy = np.concatenate([np.zeros(80), np.ones(80)])

w = np.zeros(2); b = 0.0; lr = 0.5
losses = []
for _ in range(300):
    p = 1 / (1 + np.exp(-(Xtoy @ w + b)))
    loss = -np.mean(ytoy * np.log(p + 1e-12) + (1 - ytoy) * np.log(1 - p + 1e-12))
    losses.append(float(loss))
    w -= lr * Xtoy.T @ (p - ytoy) / len(Xtoy)     # gradient step on w
    b -= lr * float(np.mean(p - ytoy))            # gradient step on b

print("loss at epoch 1 / 300:", round(losses[0], 4), "/", round(losses[-1], 4))
print("training accuracy:", float(np.mean(((Xtoy @ w + b) > 0).astype(int) == ytoy)))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_title("loss decreases as gradient descent proceeds")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy")
plt.show()

## 🎯 Problem 5.4 — The LinearClassifier class

**Given:** nothing new — you are packaging the worked example's math into a reusable class.

**Required:** write `class LinearClassifier(lr=0.1, epochs=500)` with:

- `fit(X, y, X_val=None, y_val=None)` — full-batch gradient descent from `w = 0, b = 0`; record the per-epoch training loss in `self.train_loss_` and (if validation data is passed) the validation loss in `self.val_loss_`; returns `self`,
- `predict(X)` — labels in `{0, 1}`,
- `accuracy(X, y)` — fraction correct (float).

**Expected output:** on the double moon, the training loss decreases from ~0.69 and the model settles around 80–86% train accuracy — a straight boundary can only do so much on crescents. The loss curves are the assignment's Task-2 plot.

In [ ]:
class LinearClassifier:
    """Binary logistic regression trained by full-batch gradient descent."""

    def __init__(self, lr=0.1, epochs=500):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y, X_val=None, y_val=None):
        """Fit w, b by gradient descent; record losses in train_loss_ / val_loss_."""
        # TODO: Your code here
        pass

    def predict(self, X):
        """Return predicted labels in {0, 1}."""
        # TODO: Your code here
        pass

    def accuracy(self, X, y):
        """Return the fraction of correct predictions."""
        # TODO: Your code here
        pass

# Demo call (uses your dataset + split once implemented)
demo_data34 = generate_double_moon(100, 100, seed=11)
if demo_data34 is not None:
    dX, dy = demo_data34
    dsp = train_validation_test_split(dX, dy, seed=11)
    if dsp is not None:
        (dXt, dyt), (dXv, dyv), (_, _) = dsp
        demo_lc = LinearClassifier(lr=0.1, epochs=200).fit(dXt, dyt, dXv, dyv)
        if getattr(demo_lc, "train_loss_", None) is not None:
            print("loss first/last:", round(demo_lc.train_loss_[0], 4), "/", round(demo_lc.train_loss_[-1], 4))
        else:
            print("Implement LinearClassifier.fit to see the loss curve.")
    else:
        print("Finish Problem 5.3 first to power this demo.")
else:
    print("Finish Problem 5.2 first to power this demo.")

<details>
<summary>💡 Hint 1 — lift the worked example</summary>

The Part 3 worked example IS the fit method: wrap its loop in the class, store w and b on self, record the losses every epoch (validation loss only when X_val is not None), and return self.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
n, d = X.shape
w = np.zeros(d); b = 0.0
for each epoch:
    p = sigmoid(X @ w + b)
    w -= lr * X.T @ (p - y) / n; b -= lr * mean(p - y)
    append cross-entropy (clip p to [1e-12, 1-1e-12] first) to train_loss_
    if X_val is not None: append validation cross-entropy to val_loss_
self.w, self.b = w, b; return self
predict: sigmoid(X @ w + b) >= 0.5 ; accuracy: mean(predict == y)
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
class LinearClassifier:
    """Binary logistic regression trained by full-batch gradient descent."""

    def __init__(self, lr=0.1, epochs=500):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y, X_val=None, y_val=None):
        n, d = X.shape
        w = np.zeros(d); b = 0.0
        self.train_loss_, self.val_loss_ = [], []
        for _ in range(self.epochs):
            p = 1.0 / (1.0 + np.exp(-np.clip(X @ w + b, -500, 500)))
            grad_w = X.T @ (p - y) / n
            grad_b = float(np.mean(p - y))
            w -= self.lr * grad_w
            b -= self.lr * grad_b
            p = np.clip(p, 1e-12, 1 - 1e-12)
            self.train_loss_.append(float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))))
            if X_val is not None:
                pv = np.clip(1.0 / (1.0 + np.exp(-np.clip(X_val @ w + b, -500, 500))), 1e-12, 1 - 1e-12)
                self.val_loss_.append(float(-np.mean(y_val * np.log(pv) + (1 - y_val) * np.log(1 - pv))))
        self.w, self.b = w, b
        return self

    def predict(self, X):
        return (1.0 / (1.0 + np.exp(-np.clip(X @ self.w + self.b, -500, 500))) >= 0.5).astype(int)

    def accuracy(self, X, y):
        return float(np.mean(self.predict(X) == y))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.4
data34 = generate_double_moon(240, 240, seed=42)
assert data34 is not None, "❌ generate_double_moon returned None — finish Problem 5.2 first."
X34, y34 = data34
sp34 = train_validation_test_split(X34, y34, seed=42)
assert sp34 is not None, "❌ train_validation_test_split returned None — finish Problem 5.3 first."
(Xtr, ytr), (Xv, yv), (Xte, yte) = sp34
clf = LinearClassifier(lr=0.1, epochs=400)
out34 = clf.fit(Xtr, ytr, Xv, yv)
assert out34 is not None, "❌ fit returned None — replace the 'pass'. See Hint 2 in Part 3 (and return self)."
tl = getattr(clf, "train_loss_", None)
assert tl is not None and len(tl) == 400, "❌ Record one training loss per epoch in self.train_loss_ (length 400)."
assert all(np.isfinite(tl)), "❌ Losses must be finite — clip the sigmoid argument or the log argument if you see NaN."
assert tl[-1] < tl[0], "❌ The training loss should decrease — check the gradient signs (w -= lr * grad)."
assert tl[-1] < 0.45, "❌ The final loss should drop well below the ~0.69 start — gradient descent seems stuck."
vl = getattr(clf, "val_loss_", None)
assert vl is not None and len(vl) == 400, "❌ With validation data passed, record the per-epoch validation loss in self.val_loss_."
preds = clf.predict(Xtr)
assert set(np.unique(preds)).issubset({0, 1}), "❌ predict must return labels in {0, 1} (threshold the probability at 0.5)."
acc_tr = clf.accuracy(Xtr, ytr)
assert 0.75 <= acc_tr <= 1.0, f"❌ Train accuracy should land in [0.75, 1.0], got {acc_tr:.3f} — a straight boundary on the double moon reaches the low/mid 0.8s. Check the gradient math."
assert abs(acc_tr - float(np.mean(preds == ytr))) < 1e-12, "❌ accuracy(X, y) must equal the fraction of correct predict(X) labels."
print("✅ Problem 5.4 passed — logistic regression trained.")

## Part 4: A pure-numpy neural network (MLNN) (≈15 min)

Add one **hidden layer** and the model can bend its boundary around the crescents:

- Forward: $Z_1 = XW_1 + b_1$, $A_1 = \sigma(Z_1)$, $Z_2 = A_1W_2 + b_2$, $A_2 = \sigma(Z_2)$ — the output is a probability.
- Loss: MSE, $\tfrac{1}{n}\sum_i (A_2 - y_i)^2$ (the assignment's choice).
- Backward (chain rule): $dZ_2 = \tfrac{2}{n}(A_2 - y)\,A_2(1 - A_2)$, then $dW_2 = A_1^\top dZ_2$, $dZ_1 = (dZ_2 \otimes W_2)\odot A_1(1 - A_1)$, $dW_1 = X^\top dZ_1$ — plus the bias sums.
- **Momentum**: $v \leftarrow \mu v - \text{lr}\,\nabla$, $W \leftarrow W + v$ — accelerates along steady descent directions and damps oscillation.
- Initialize $W \sim \mathcal{N}(0, \sqrt{1/\text{fan\_in}})$: small random weights break symmetry, and the scale keeps sigmoids in their lively region.

In [ ]:
# --- Worked example: forward pass of a tiny network — shape by shape ---
rng = np.random.default_rng(0)
X_demo = np.array([[0.0, 0.0], [1.0, -1.0]])          # 2 samples, 2 features
W1 = rng.normal(0, np.sqrt(1 / 2), size=(2, 4)); b1 = np.zeros(4)
W2 = rng.normal(0, np.sqrt(1 / 4), size=4);      b2 = 0.0

A1 = 1 / (1 + np.exp(-(X_demo @ W1 + b1)))          # (2, 4): hidden activations
A2 = 1 / (1 + np.exp(-(A1 @ W2 + b2)))              # (2,):  output probabilities
print("X:", X_demo.shape, "-> A1:", A1.shape, "-> A2:", A2.shape)
print("outputs:", np.round(A2, 3), " <- probabilities in (0, 1); training adjusts W1, W2 so these match y")

## 🎯 Problem 5.5 — The MLNN class

**Given:** the architecture above.

**Required:** write `class MLNN(hidden=24, lr=0.2, epochs=1500, momentum=0.9, seed=0)` with `fit` / `predict` / `accuracy` and per-epoch MSE `train_loss_` / `val_loss_`, matching the assignment's signature:

- $W_1 \sim \mathcal{N}(0, \sqrt{1/d})$ of shape $(d, \text{hidden})$, $b_1 = 0$; $W_2 \sim \mathcal{N}(0, \sqrt{1/\text{hidden}})$ of shape $(\text{hidden},)$, $b_2 = 0$ — all drawn from `np.random.default_rng(seed)`,
- $dZ_2 = \tfrac{2}{n}(A_2 - y)\,A_2(1 - A_2)$; $dW_2 = A_1^\top dZ_2$; $db_2 = \sum dZ_2$; $dZ_1 = (dZ_2 \otimes W_2)\odot A_1(1 - A_1)$; $dW_1 = X^\top dZ_1$; $db_1 = \sum_{\text{rows}} dZ_1$,
- momentum $\mu = 0.9$ on every parameter; predict: $A_2 \geq 0.5$.

**Expected output:** on the double moon the loss falls from ~0.3 to under 0.1 and test accuracy reaches the mid/high 0.9s — clearly beating the linear model.

In [ ]:
class MLNN:
    """One-hidden-layer MLP: 2 -> hidden -> 1, sigmoid activations, MSE loss (pure NumPy)."""

    def __init__(self, hidden=24, lr=0.2, epochs=1500, momentum=0.9, seed=0):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.momentum = momentum
        self.seed = seed

    def fit(self, X, y, X_val=None, y_val=None):
        """Train the network; record MSE losses in train_loss_ / val_loss_."""
        # TODO: Your code here (forward pass, backpropagation, momentum update)
        pass

    def predict(self, X):
        """Return predicted labels in {0, 1}."""
        # TODO: Your code here
        pass

    def accuracy(self, X, y):
        """Return the fraction of correct predictions."""
        # TODO: Your code here
        pass

# Demo call (uses your dataset + split once implemented)
demo_data35 = generate_double_moon(100, 100, seed=13)
if demo_data35 is not None:
    nX, ny = demo_data35
    nsp = train_validation_test_split(nX, ny, seed=13)
    if nsp is not None:
        (nXt, nyt), (nXv, nyv), (_, _) = nsp
        demo_nn = MLNN(hidden=12, epochs=300).fit(nXt, nyt, nXv, nyv)
        if getattr(demo_nn, "train_loss_", None) is not None:
            print("MLNN loss first/last:", round(demo_nn.train_loss_[0], 4), "/", round(demo_nn.train_loss_[-1], 4))
        else:
            print("Implement MLNN.fit to see the loss curve.")
    else:
        print("Finish Problem 5.3 first to power this demo.")
else:
    print("Finish Problem 5.2 first to power this demo.")

<details>
<summary>💡 Hint 1 — forward, backward, momentum</summary>

Three blocks per epoch: (1) forward pass storing A1, A2; append MSE to train_loss_; (2) backward chain dZ2 -> dW2/db2 -> dZ1 -> dW1/db1 (include the 2/n factor in dZ2); (3) momentum velocity update for all four parameters, then W += v.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
init W1 ~ N(0, sqrt(1/d)) (d, hidden), b1 = 0, W2 ~ N(0, sqrt(1/hidden)) (hidden,), b2 = 0
loop epochs:
    A1 = sigmoid(X @ W1 + b1); A2 = sigmoid(A1 @ W2 + b2)
    append mean((A2 - y)**2) to train_loss_ (same for val if given)
    dZ2 = 2 * (A2 - y) * A2 * (1 - A2) / n
    dW2 = A1.T @ dZ2; db2 = sum(dZ2)
    dZ1 = outer(dZ2, W2) * A1 * (1 - A1)
    dW1 = X.T @ dZ1; db1 = dZ1.sum(axis=0)
    v = momentum * v - lr * grad for each param; W += v
predict: A2 >= 0.5
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
class MLNN:
    """One-hidden-layer MLP: 2 -> hidden -> 1, sigmoid activations, MSE loss (pure NumPy)."""

    def __init__(self, hidden=24, lr=0.2, epochs=1500, momentum=0.9, seed=0):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.momentum = momentum
        self.seed = seed

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        W1 = rng.normal(0, np.sqrt(1 / d), size=(d, self.hidden)); b1 = np.zeros(self.hidden)
        W2 = rng.normal(0, np.sqrt(1 / self.hidden), size=self.hidden); b2 = 0.0
        vW1 = np.zeros_like(W1); vb1 = np.zeros_like(b1)
        vW2 = np.zeros_like(W2); vb2 = 0.0
        self.train_loss_, self.val_loss_ = [], []
        for _ in range(self.epochs):
            Z1 = X @ W1 + b1
            A1 = 1.0 / (1.0 + np.exp(-np.clip(Z1, -500, 500)))
            Z2 = A1 @ W2 + b2
            A2 = 1.0 / (1.0 + np.exp(-np.clip(Z2, -500, 500)))
            self.train_loss_.append(float(np.mean((A2 - y) ** 2)))
            if X_val is not None:
                Av = 1.0 / (1.0 + np.exp(-np.clip(X_val @ W1 + b1, -500, 500)))
                A2v = 1.0 / (1.0 + np.exp(-np.clip(Av @ W2 + b2, -500, 500)))
                self.val_loss_.append(float(np.mean((A2v - y_val) ** 2)))
            dZ2 = 2 * (A2 - y) * A2 * (1 - A2) / n
            dW2 = A1.T @ dZ2; db2 = float(np.sum(dZ2))
            dA1 = np.outer(dZ2, W2); dZ1 = dA1 * A1 * (1 - A1)
            dW1 = X.T @ dZ1; db1 = dZ1.sum(axis=0)
            vW1 = self.momentum * vW1 - self.lr * dW1; vb1 = self.momentum * vb1 - self.lr * db1
            vW2 = self.momentum * vW2 - self.lr * dW2; vb2 = self.momentum * vb2 - self.lr * db2
            W1 += vW1; b1 += vb1; W2 += vW2; b2 += vb2
        self.W1, self.b1, self.W2, self.b2 = W1, b1, W2, b2
        return self

    def predict(self, X):
        A1 = 1.0 / (1.0 + np.exp(-np.clip(X @ self.W1 + self.b1, -500, 500)))
        A2 = 1.0 / (1.0 + np.exp(-np.clip(A1 @ self.W2 + self.b2, -500, 500)))
        return (A2 >= 0.5).astype(int)

    def accuracy(self, X, y):
        return float(np.mean(self.predict(X) == y))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.5
data35 = generate_double_moon(240, 240, seed=42)
assert data35 is not None, "❌ generate_double_moon returned None — finish Problem 5.2 first."
X35, y35 = data35
sp35 = train_validation_test_split(X35, y35, seed=42)
assert sp35 is not None, "❌ train_validation_test_split returned None — finish Problem 5.3 first."
(Xtr5n, ytr5n), (Xv5n, yv5n), (Xte5n, yte5n) = sp35
net = MLNN(hidden=16, lr=0.2, epochs=600, momentum=0.9, seed=0)
out35 = net.fit(Xtr5n, ytr5n, Xv5n, yv5n)
assert out35 is not None, "❌ fit returned None — replace the 'pass'. See Hint 2 in Part 4 (and return self)."
tl5 = getattr(net, "train_loss_", None)
assert tl5 is not None and len(tl5) == 600, "❌ Record one MSE training loss per epoch in self.train_loss_."
assert all(np.isfinite(tl5)), "❌ Losses must be finite — clip sigmoid arguments (np.clip(z, -500, 500))."
assert tl5[-1] < 0.5 * tl5[0], "❌ The MSE loss should at least halve over training — check the backprop chain (dZ2 first, then dZ1)."
vl5 = getattr(net, "val_loss_", None)
assert vl5 is not None and len(vl5) == 600, "❌ Record the per-epoch validation MSE in self.val_loss_."
pr5 = net.predict(Xte5n)
assert set(np.unique(pr5)).issubset({0, 1}), "❌ predict must return labels in {0, 1}."
acc5 = net.accuracy(Xte5n, yte5n)
assert acc5 >= 0.9, f"❌ Test accuracy should reach >= 0.9, got {acc5:.3f} — a hidden layer should clearly beat the ~0.82 linear model. Check the gradient scaling (the 2/n factor) and the momentum update."
assert net.accuracy(Xtr5n, ytr5n) >= 0.9, "❌ The network should fit the training set well above 0.9 — training is not converging."
print("✅ Problem 5.5 passed — the neural network learns the crescents.")

## 🎯 Problem 5.6 — Mini-challenge: the full comparison

**Given:** your dataset, split, `LinearClassifier` and `MLNN`.

**Required:** write `compare_classifiers(N1=200, N2=200, seed=7)` that:

1. generates and splits the double moon (same `seed` for both),
2. trains a linear model and an MLNN (`hidden=24`, `epochs=1500`) on the **same** training set,
3. builds a decision-boundary figure: a 1×2 subplot grid; for each model, `predict` over a fine grid covering the data (`np.meshgrid` on the axis ranges ±1), `contourf` the predicted labels, and scatter the training points colored by class,
4. returns a dict with keys `'linear_test_acc'`, `'mlnn_test_acc'`, `'better'` (`"MLNN"` if the MLNN test accuracy is higher, else `"Linear"`) and `'figure'`.

**Expected output:** the linear panel shows a straight boundary (~0.86 test accuracy), the MLNN panel a curved one (~0.98) — the exact comparison **Lab Assignment 05** asks for in its Task 4.

In [ ]:
def compare_classifiers(N1=200, N2=200, seed=7):
    """Train both classifiers on one split; return accuracies + boundary figure.

    Args:
        N1, N2: points per class
        seed: reproducibility seed for generation and splitting

    Returns:
        dict with keys 'linear_test_acc', 'mlnn_test_acc', 'better', 'figure'
    """
    # TODO: Your code here
    pass

# Demo call
demo_cmp = compare_classifiers()
if demo_cmp is not None:
    print("linear:", round(demo_cmp["linear_test_acc"], 3),
          "| MLNN:", round(demo_cmp["mlnn_test_acc"], 3),
          "| better:", demo_cmp["better"])
    plt.show()
else:
    print("Implement compare_classifiers to power this demo.")

<details>
<summary>💡 Hint 1 — same split, grid, contourf</summary>

Generate once, split once, train both models on the same training set. For the figure: meshgrid over [min-1, max+1] of each axis, ravel to points, predict, reshape back, contourf with levels [-0.5, 0.5, 1.5], scatter the training points, title with the test accuracy.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
X, y = generate_double_moon(N1, N2, seed=seed)
split once with the same seed
lin = LinearClassifier(...).fit(X_train, ...)
net = MLNN(hidden=24, epochs=1500).fit(X_train, ...)
GX, GY = np.meshgrid(linspace ranges +-1)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
per model: contourf(GX, GY, model.predict(grid_pts).reshape(GX.shape))
            scatter(X_train, c=y_train); title with model.accuracy(X_test, y_test)
return dict with both test accuracies, the verdict and the figure
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def compare_classifiers(N1=200, N2=200, seed=7):
    """Train both classifiers on one split; return accuracies + boundary figure."""
    X, y = generate_double_moon(N1, N2, seed=seed)
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = train_validation_test_split(X, y, seed=seed)

    lin = LinearClassifier(lr=0.1, epochs=500).fit(X_train, y_train, X_val, y_val)
    net = MLNN(hidden=24, lr=0.2, epochs=1500).fit(X_train, y_train, X_val, y_val)

    gx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    gy = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
    GX, GY = np.meshgrid(gx, gy)
    grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, model, name in [(axes[0], lin, "linear"), (axes[1], net, "MLNN")]:
        P = model.predict(grid_pts).reshape(GX.shape)
        ax.contourf(GX, GY, P, levels=[-0.5, 0.5, 1.5], alpha=0.3, cmap="bwr")
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=6, cmap="bwr")
        ax.set_title(f"{name}: test acc = {model.accuracy(X_test, y_test):.3f}")
    fig.tight_layout()

    lin_acc = float(lin.accuracy(X_test, y_test))
    net_acc = float(net.accuracy(X_test, y_test))
    return {"linear_test_acc": lin_acc, "mlnn_test_acc": net_acc,
            "better": "MLNN" if net_acc > lin_acc else "Linear", "figure": fig}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 5.6
cmp_res = compare_classifiers()
assert cmp_res is not None, "❌ compare_classifiers returned None — replace the 'pass'. See Hint 2 in Part 4."
for k in ("linear_test_acc", "mlnn_test_acc", "better", "figure"):
    assert k in cmp_res, f"❌ Dict is missing '{k}'."
assert 0.0 <= cmp_res["linear_test_acc"] <= 1.0 and 0.0 <= cmp_res["mlnn_test_acc"] <= 1.0, "❌ Accuracies must be fractions in [0, 1]."
expected_better = "MLNN" if cmp_res["mlnn_test_acc"] > cmp_res["linear_test_acc"] else "Linear"
assert cmp_res["better"] == expected_better, "❌ 'better' must be 'MLNN' exactly when the MLNN test accuracy is higher."
assert cmp_res["mlnn_test_acc"] >= cmp_res["linear_test_acc"] - 0.05, "❌ The MLNN should not lose to the linear model by more than 0.05 — check that both train on the SAME split and that the MLNN uses momentum."
import matplotlib.figure as _mpl_fig5
import matplotlib.collections as _mpl_col5
from matplotlib.contour import QuadContourSet as _QCS5

def _has_contourf5(ax):
    # matplotlib >= 3.8 registers contourf as a QuadContourSet in ax.collections;
    # older versions register a QuadMesh. Accept either.
    return any(isinstance(c, (_mpl_col5.QuadMesh, _QCS5)) for c in ax.collections)

fig5 = cmp_res["figure"]
assert isinstance(fig5, _mpl_fig5.Figure), "❌ 'figure' must be a matplotlib Figure."
panels5 = fig5.get_axes()[:2]
assert len(panels5) == 2, "❌ The figure needs 2 side-by-side boundary panels."
for i, ax in enumerate(panels5):
    assert _has_contourf5(ax), f"❌ Panel {i} needs a contourf of the predicted labels over the grid."
    assert len(ax.collections) >= 2, f"❌ Panel {i} should show both the decision region and the scattered training points."
print("✅ Problem 5.6 passed — linear vs neural, side by side. Lab Assignment 05 awaits!")

## 🎉 You've completed Lab Activity 5

You have mastered:
- Generating the double moon geometrically: half-annulus crescents with radial band $[\text{radius} - \text{width}/2, \text{radius} + \text{width}/2]$, seeded for reproducibility
- Shuffling and splitting 60/20/20 with one seeded permutation — disjoint, exhaustive, roughly stratified
- Training logistic regression with full-batch gradient descent and reading its loss curve
- Training a one-hidden-layer pure-numpy network (sigmoid, MSE, momentum) — and why it beats the straight boundary
- Comparing models on the same split: accuracies, verdict, and a decision-boundary figure

**You are now ready for Lab Assignment 05 on the course portal — the assignment asks for the same techniques without hints.**

💡 **Tip:** restart the kernel and run every cell top-to-bottom once more — each 🧪 self-check should print ✅. If you have time, retrain the MLNN with `hidden=8` and watch the boundary get less flexible: capacity is a dial, not a switch.